In [1]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import os
import cv2
import json
import glob
from tqdm import tqdm

In [2]:
class MBH_dataset:
    def __init__(self, data_dir, label_file, transform):
        self.data_dir = data_dir
        self.label_file = label_file
        with open(self.label_file) as f:
            self.text = [line for line in f]
        self.image_paths = []
        self.labels = []
        self.transform = None
        self.ok_image_files = []  # Initialize as instance attributes
        self.nok_image_files = []  # Initialize as instance attributes

    def load_data(self):
        for lines in self.text:
            temp = str(lines).split()
            self.img_name = temp[0]
            self.img_folder = temp[1]
            self.label = temp[2]
            image_path = self.data_dir + "total_data/" + str(self.img_folder) + "/" + self.img_name
            self.image_paths.append(image_path)
            self.labels.append(int(self.label))
            
            if int(self.label) == 0:
                self.ok_image_files.append(image_path)
            else:
                self.nok_image_files.append(image_path)

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        image = cv.imread(image_path)
        label = self.labels[index]
        

        if self.transform:
            image = self.transform(image)

        return image, label

        return ok_image_files, nok_image_files

In [3]:
dataset = MBH_dataset(data_dir = "C:/Users/resu/Desktop/FAPS - AI Project/datasets/Fangjun_Wang_2021_MBH-dataset/MBH-dataset/", 
                     label_file = "C:/Users/resu/Desktop/Project - II Final Submission/Chapter 3 - Data Understanding and Preparation/Labels/Part - E/train.txt", transform = None)

In [4]:
dataset.load_data()  # Load the data
#print(dataset.ok_image_files)  # Access ok_image_files
ok_dict = {element: 'okay' for element in dataset.ok_image_files}
nok_dict = {element: 'not okay' for element in dataset.nok_image_files}

ok_dict.update(nok_dict)
labels = ok_dict


files = labels.keys()

dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
dinov2_vits14.to(device)

transform_image = T.Compose([T.ToTensor(), T.Resize(244), T.CenterCrop(224), T.Normalize([0.5], [0.5])])


def load_image(img: str) -> torch.Tensor:
    """
    Load an image and return a tensor that can be used as an input to DINOv2.
    """
    img = Image.open(img)

    transformed_img = transform_image(img)[:3].unsqueeze(0)

    return transformed_img



def compute_embeddings(files: list) -> dict:
    """
    Create an index that contains all of the images in the specified list of files.
    """
    all_embeddings = {}

    with torch.no_grad():
        for i, file in enumerate(tqdm(files)):
            embeddings = dinov2_vits14(load_image(file).to(device))

            all_embeddings[file] = np.array(embeddings[0].cpu().numpy()).reshape(1, -1).tolist()

    with open("part_e.json", "w") as f:
        f.write(json.dumps(all_embeddings))

    return all_embeddings


embeddings = compute_embeddings(files)

from sklearn import svm

clf = svm.SVC(gamma='scale')
y = [labels[file] for file in files]
embedding_list = list(embeddings.values())
clf.fit(np.array(embedding_list).reshape(-1, 384), y)




Using cache found in C:\Users\resu/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\resu/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\resu/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\resu/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
100%|██████████████████████████████████████████████████████████████████████████████| 2529/2529 [05:11<00:00,  8.12it/s]


SVC()

In [5]:
###Example 

'''import supervision as sv

input_file = "C:/Users/resu/Desktop/FAPS - AI Project/datasets/Fangjun_Wang_2021_MBH-dataset/MBH-dataset/total_data/Part-0005/0149@View-001@2020-11-24_13-19-04.jpg_Part-0005.jpg"

new_image = load_image(input_file)

%matplotlib inline
sv.plot_image(image=cv2.imread(input_file), size=(8, 8))

with torch.no_grad():
    embedding = dinov2_vits14(new_image.to(device))

    prediction = clf.predict(np.array(embedding[0].cpu()).reshape(1, -1))

    print()
    print("Predicted class: " + prediction[0])'''

'import supervision as sv\n\ninput_file = "C:/Users/resu/Desktop/FAPS - AI Project/datasets/Fangjun_Wang_2021_MBH-dataset/MBH-dataset/total_data/Part-0005/0149@View-001@2020-11-24_13-19-04.jpg_Part-0005.jpg"\n\nnew_image = load_image(input_file)\n\n%matplotlib inline\nsv.plot_image(image=cv2.imread(input_file), size=(8, 8))\n\nwith torch.no_grad():\n    embedding = dinov2_vits14(new_image.to(device))\n\n    prediction = clf.predict(np.array(embedding[0].cpu()).reshape(1, -1))\n\n    print()\n    print("Predicted class: " + prediction[0])'

In [6]:
test_dataset = MBH_dataset(data_dir = "C:/Users/resu/Desktop/FAPS - AI Project/datasets/Fangjun_Wang_2021_MBH-dataset/MBH-dataset/", 
                     label_file = "C:/Users/resu/Desktop/Project - II Final Submission/Chapter 3 - Data Understanding and Preparation/Labels/Part - E/model_test.txt", transform = None)



test_dataset.load_data() 
test_ok_dict = {element: 'okay' for element in test_dataset.ok_image_files}
test_nok_dict = {element: 'not okay' for element in test_dataset.nok_image_files}


test_ok_dict.update(test_nok_dict)

labels_test = test_ok_dict


import supervision as sv

labels_test_key = list(labels_test.keys())

predicted_test_key = [] 

for i in range(len(labels_test_key)):
    input_file = labels_test_key[i]

    new_image = load_image(input_file)

    %matplotlib inline
    #sv.plot_image(image=cv2.imread(input_file), size=(8, 8))

    with torch.no_grad():
        embedding = dinov2_vits14(new_image.to(device))

        prediction = clf.predict(np.array(embedding[0].cpu()).reshape(1, -1))

        print()
        print("Predicted class: " + prediction[0])
        predicted_test_key.append(prediction[0])
        


Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted class: okay

Predicted 


Predicted class: okay

Predicted class: okay

Predicted class: not okay


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/resu/Desktop/FAPS - AI Project/datasets/Fangjun_Wang_2021_MBH-dataset/MBH-dataset/total_data/Part-0014/0915@View-001@2020-12-5_14-16-33.jpg_Part-0014.jpg'

In [7]:
TP = 0 
TN = 0 
FP = 0 
FN = 0 

labels_test_val = list(labels_test.values())


for i in range(len(labels_test_val)):
    if (labels_test_val[i] == 'okay' and predicted_test_key[i] == "okay"):
        TP+=1
    elif labels_test_val[i] == "not_okay" and predicted_test_key[i]=="not_okay":
        TN+=1 
    elif labels_test_val[i]=="not_okay" and predicted_test_key[i]=="okay":
        FP+=1 
    elif labels_test_val[i]=="okay" and predicted_test_key[i]=="not_okay":
        FN+=1 


print("TP:", TP, "\tFP:", FP, "\nFN:", FN, "\tTN:", TN)
        

TP: 298 	FP: 0 
FN: 0 	TN: 0


In [8]:
len(ok_dict)

2529